In [ ]:
!pip install autogluon.tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
datasetPath = Path("/content/drive/MyDrive/ML-For-CV-Robustness/datasets/datasetGTCombined.csv")

#making sure
datasetPath.is_file()

True

In [ ]:
dataset = pd.read_csv(datasetPath)
dataset

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
667235,0.261726,0.000240,0.000362,553.313000,483.020000,15.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667236,0.239247,0.000270,0.000396,553.313000,483.020000,16.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667237,0.283672,0.000269,0.000422,553.313000,483.020000,17.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667238,0.241322,0.000345,0.000508,553.313000,483.020000,18.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26


In [ ]:
from autogluon.tabular import TabularPredictor

In [ ]:
train_df = dataset.sample(frac=0.8, random_state=42)
test_df = dataset.drop(train_df.index)

In [ ]:
train_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
239397,0.255627,0.000572,0.000916,262.126637,200.472881,17.0,180.0,-19.763107,-104.398338,-0.0,-19.620,-0.00,0.32
460616,0.291634,0.000310,0.000475,334.972000,302.575000,16.0,180.0,-63.812729,-56.074120,0.0,-22.440,-147.65,0.30
193058,0.280101,0.000411,0.000702,1336.747674,1022.588409,18.0,180.0,-19.798325,104.358398,-0.0,-19.278,-0.00,0.36
555725,0.811764,0.000306,0.000472,223.838000,202.044000,5.0,180.0,-63.688171,56.363342,-0.0,-22.038,-180.00,0.36
335353,0.180071,0.029525,0.037890,1.540000,2.103000,13.0,180.0,-63.812729,-56.074120,0.0,-22.053,180.00,0.36
...,...,...,...,...,...,...,...,...,...,...,...,...,...
539354,0.188815,0.000378,0.000529,1174.170000,1024.945000,14.0,180.0,-42.301559,-85.835487,-0.0,-19.350,-0.00,0.26
651738,0.152599,0.041819,0.054116,2.088000,2.497000,18.0,180.0,-19.798325,104.358398,-0.0,-19.250,-0.00,0.38
512486,0.205018,0.000227,0.000300,1138.042000,1026.515000,6.0,180.0,-63.688171,56.363342,-0.0,-19.670,-0.00,0.38
185954,0.197553,0.000359,0.000502,1165.530874,1015.519825,14.0,180.0,-42.151062,85.965363,-0.0,-19.476,-0.00,0.26


In [ ]:
test_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
5,0.081775,0.000463,0.000538,1130.973355,1020.232214,5.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
8,0.132084,0.000445,0.000558,1130.973355,1020.232214,8.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
13,0.108716,0.000371,0.000444,1130.973355,1020.232214,13.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
15,0.144419,0.000328,0.000411,1130.973355,1020.232214,15.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...
667222,0.258746,0.000165,0.000226,553.313000,483.020000,2.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667223,0.261627,0.000149,0.000207,553.313000,483.020000,3.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667224,0.289089,0.000165,0.000243,553.313000,483.020000,4.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26
667234,0.251055,0.000267,0.000392,553.313000,483.020000,14.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.26


In [ ]:
predictor = TabularPredictor(
    label="grviOut",
    problem_type="regression",
    eval_metric="root_mean_squared_error"
).fit(
    train_data=train_df
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260909_172710"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.72 GB / 12.67 GB (84.6%)
Disk Space Avail:   65.06 GB / 112.64 GB (57.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and t

[1000]	valid_set's rmse: 0.0376263
[2000]	valid_set's rmse: 0.0343375
[3000]	valid_set's rmse: 0.0324195
[4000]	valid_set's rmse: 0.0310654
[5000]	valid_set's rmse: 0.0299647
[6000]	valid_set's rmse: 0.0291136
[7000]	valid_set's rmse: 0.0284022
[8000]	valid_set's rmse: 0.0277392
[9000]	valid_set's rmse: 0.0271678
[10000]	valid_set's rmse: 0.0267023


	-0.0267	 = Validation score   (-root_mean_squared_error)
	322.14s	 = Training   runtime
	5.6s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=1, gpus=0, mem=0.3/10.5 GB


[1000]	valid_set's rmse: 0.0267062
[2000]	valid_set's rmse: 0.0235611
[3000]	valid_set's rmse: 0.0216981
[4000]	valid_set's rmse: 0.0205361
[5000]	valid_set's rmse: 0.0196978
[6000]	valid_set's rmse: 0.0190799
[7000]	valid_set's rmse: 0.0186079
[8000]	valid_set's rmse: 0.0181736
[9000]	valid_set's rmse: 0.0178348
[10000]	valid_set's rmse: 0.0175412


	-0.0175	 = Validation score   (-root_mean_squared_error)
	214.29s	 = Training   runtime
	3.82s	 = Validation runtime
Fitting model: RandomForestMSE ...
	Fitting with cpus=2, gpus=0, mem=2.6/10.5 GB
	-0.0204	 = Validation score   (-root_mean_squared_error)
	1154.52s	 = Training   runtime
	0.78s	 = Validation runtime
Fitting model: CatBoost ...
	Fitting with cpus=1, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.6.1`.
Fitting model: ExtraTreesMSE ...
	Fitting with cpus=2, gpus=0, mem=2.6/10.5 GB
	-0.0094	 = Validation score   (-root_mean_squared_error)
	177.79s	 = Training   runtime
	0.33s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Fitting with cpus=1, gpus=0, mem=0.4/10.3 GB
	-0.0447	 = Validation score   (-root_mean_squared_error)
	328.15s	 = Training   runtime
	0.05s	 = Validation runtime
Fitting model: XGBoost ...
	Fitting with cpus=1, gpus=0
	-0.0165	 = Validation score   (-root_mean_squared_error)
	450.23

[1000]	valid_set's rmse: 0.0223759
[2000]	valid_set's rmse: 0.0193432
[3000]	valid_set's rmse: 0.0179734
[4000]	valid_set's rmse: 0.0171984
[5000]	valid_set's rmse: 0.0165874
[6000]	valid_set's rmse: 0.0161695
[7000]	valid_set's rmse: 0.0158828
[8000]	valid_set's rmse: 0.0155946
[9000]	valid_set's rmse: 0.0153722
[10000]	valid_set's rmse: 0.0151707


	-0.0152	 = Validation score   (-root_mean_squared_error)
	243.9s	 = Training   runtime
	5.28s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/10.1 GB
	Ensemble Weights: {'ExtraTreesMSE': 0.947, 'LightGBMLarge': 0.053}
	-0.0094	 = Validation score   (-root_mean_squared_error)
	0.04s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 6927.25s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 950.8 rows/s (5338 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20260909_172710")


In [ ]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.009387,root_mean_squared_error,5.614129,421.728588,0.000996,0.037460,2,True,9
1,ExtraTreesMSE,-0.009401,root_mean_squared_error,0.331450,177.792280,0.331450,177.792280,1,True,4
2,LightGBMLarge,-0.015171,root_mean_squared_error,5.281684,243.898849,5.281684,243.898849,1,True,8
3,XGBoost,-0.016460,root_mean_squared_error,1.311615,450.226148,1.311615,450.226148,1,True,6
4,LightGBM,-0.017541,root_mean_squared_error,3.815534,214.290755,3.815534,214.290755,1,True,2
5,RandomForestMSE,-0.020443,root_mean_squared_error,0.783411,1154.516738,0.783411,1154.516738,1,True,3
6,LightGBMXT,-0.026702,root_mean_squared_error,5.600628,322.135270,5.600628,322.135270,1,True,1
7,NeuralNetTorch,-0.030856,root_mean_squared_error,0.028236,3997.989761,0.028236,3997.989761,1,True,7
8,NeuralNetFastAI,-0.044681,root_mean_squared_error,0.048497,328.153593,0.048497,328.153593,1,True,5


In [ ]:
predictor.leaderboard(test_df, silent=True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,ExtraTreesMSE,-0.009429,-0.009401,root_mean_squared_error,4.274330,0.331450,177.792280,4.274330,0.331450,177.792280,1,True,4
1,WeightedEnsemble_L2,-0.009435,-0.009387,root_mean_squared_error,131.802122,5.614129,421.728588,0.025081,0.000996,0.037460,2,True,9
2,LightGBMLarge,-0.016003,-0.015171,root_mean_squared_error,127.502712,5.281684,243.898849,127.502712,5.281684,243.898849,1,True,8
3,XGBoost,-0.016860,-0.016460,root_mean_squared_error,35.568143,1.311615,450.226148,35.568143,1.311615,450.226148,1,True,6
4,LightGBM,-0.018682,-0.017541,root_mean_squared_error,96.284559,3.815534,214.290755,96.284559,3.815534,214.290755,1,True,2
5,RandomForestMSE,-0.021035,-0.020443,root_mean_squared_error,5.706963,0.783411,1154.516738,5.706963,0.783411,1154.516738,1,True,3
6,LightGBMXT,-0.027782,-0.026702,root_mean_squared_error,127.393805,5.600628,322.135270,127.393805,5.600628,322.135270,1,True,1
7,NeuralNetTorch,-0.031512,-0.030856,root_mean_squared_error,0.557021,0.028236,3997.989761,0.557021,0.028236,3997.989761,1,True,7
8,NeuralNetFastAI,-0.044638,-0.044681,root_mean_squared_error,0.924030,0.048497,328.153593,0.924030,0.048497,328.153593,1,True,5
